# ViHSD Mixture of Experts experiment (Kaggle Edition)

This notebook prepares a **Kaggle Notebook runtime** to train and evaluate the ViHSD Mixture of Experts (MoE) architectures.

Training and evaluation are executed explicitly from shell commands. This keeps experiment execution reproducible, parameter overrides transparent, and run outputs systematically tracked by unique `run_id`s.

### Essential Kaggle Settings Checklist
Before running any cells, make sure these settings are enabled in the Kaggle right-hand **Notebook settings** sidebar:
1. **Accelerator**: Select **GPU P100** (single 16GB GPU) or **GPU T4 x2** (15GB VRAM per card).
2. **Internet**: Toggle **Internet ON** (requires SMS verification on Kaggle). Internet is required to clone this repository, download the ViHSD dataset from Hugging Face, download PhoBERT weights, and log to Weights & Biases.
3. **Secrets**: Under the notebook's top menu, open **Add-ons → Secrets** and create:
   - `HF_TOKEN`: Hugging Face access token.
   - `WANDB_API_KEY`: Weights & Biases API key.
4. **Output Persistence**: Everything written to `/kaggle/working` is saved permanently when you click **"Save Version" → "Save & Run All (Commit)"**. Completed runs will appear in the notebook's **Output** tab.

---

## 1. Configure Kaggle Output Directories

On Kaggle, `/kaggle/working` is the only writable directory during execution. We configure `CHECKPOINT_DIR` and `RESULTS_DIR` environment variables so all model checkpoints (`.safetensors`) and experiment logs (`run_metrics.json`, `vihsd_predictions.json`) are saved directly to `/kaggle/working/checkpoints` and `/kaggle/working/results`.

In [ ]:
import os
from pathlib import Path

checkpoint_dir = Path('/kaggle/working/checkpoints')
results_dir = Path('/kaggle/working/results')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

os.environ['CHECKPOINT_DIR'] = str(checkpoint_dir)
os.environ['RESULTS_DIR'] = str(results_dir)

print(f"CHECKPOINT_DIR: {os.environ['CHECKPOINT_DIR']}")
print(f"RESULTS_DIR:    {os.environ['RESULTS_DIR']}")

## 2. Clone the Repository & Verify Dependencies

This step ensures the repository code is available in the workspace and targets the `main` branch.

If you are running this notebook inside an already cloned or uploaded workspace where `train.py` is present, it will automatically use the current directory without re-cloning.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/kaggle/working/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'
BRANCH = 'main'

if not Path('train.py').exists():
    if not Path(PROJECT_DIR).exists():
        print(f"Cloning {REPOSITORY_URL} (branch: {BRANCH})...")
        !git clone --depth 1 --branch $BRANCH $REPOSITORY_URL $PROJECT_DIR
    %cd $PROJECT_DIR
else:
    print(f"Already in project repository root: {Path.cwd()}")

%pip install -q -r requirements.txt

## 3. Authenticate via Kaggle Secrets

Authentication credentials for Hugging Face and Weights & Biases are retrieved securely using Kaggle's `UserSecretsClient`.

Ensure you have added `HF_TOKEN` and `WANDB_API_KEY` under **Add-ons → Secrets** in the notebook toolbar.

In [ ]:
import os
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
except Exception as error:
    print(f"Kaggle secrets client notice: {error}")
    hf_token = os.getenv('HF_TOKEN')
    wandb_api_key = os.getenv('WANDB_API_KEY')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("✓ Hugging Face authentication configured.")
else:
    print("! Notice: HF_TOKEN secret not found; public Hugging Face assets will still load.")

if wandb_api_key:
    os.environ['WANDB_API_KEY'] = wandb_api_key
    wandb.login(key=wandb_api_key)
    print("✓ Weights & Biases authentication successful.")
else:
    print("! Notice: WANDB_API_KEY secret not found; set logging.use_wandb=false if W&B is not needed.")

## 4. Run Training and Evaluation Manually

The comparison uses two architectures: `dense_phobert` and `phobert_moe`. Each run saves its resolved configuration, architecture-specific checkpoint, metrics, and predictions. Use a unique `--run-id` for every experiment.

### Smoke test

```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Dense PhoBERT baseline

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id dense-phobert-integrated \
  --set model.architecture=dense_phobert \
  --set model.pooling=mean \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id dense-phobert-integrated
```

### PhoBERT MoE

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id phobert-moe-integrated \
  --set model.architecture=phobert_moe \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id phobert-moe-integrated
```

When debugging, inspect `checkpoints/<run-id>/resolved_config.yaml` before rerunning.

## 5. Training Command

Edit the command below and execute the cell manually. For a persistent run, save a Kaggle version via **Save Version → Save & Run All (Commit)**.

In [ ]:
# Edit the run_id before executing this cell.
# Dense PhoBERT baseline:
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id dense-phobert-integrated --set model.architecture=dense_phobert --set model.pooling=mean --set training.loss_type=cross_entropy

# PhoBERT MoE:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id phobert-moe-integrated --set model.architecture=phobert_moe --set training.loss_type=cross_entropy

## 6. Evaluation Command

Run evaluation after training finishes. Pass the matching `--run-id` to load that run's best checkpoint (`vihsd_moe_best.safetensors`) and compute test set metrics.

In [ ]:
# Use the run_id from the training command above:
!python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe

# Or evaluate a specific checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint /kaggle/working/checkpoints/baseline-current-moe/vihsd_moe_best.safetensors

## 7. Command-Line Argument Reference

### Supported architectures

| Architecture | Purpose |
| :--- | :--- |
| `dense_phobert` | Dense PhoBERT baseline with mean pooling. |
| `phobert_moe` | PhoBERT with token-level MoE layers. |

For dense PhoBERT, always pass `--set model.pooling=mean`. Evaluation uses the same `--run-id` and automatically finds the architecture-specific checkpoint. Existing legacy checkpoints are still supported.

### Common settings

| Section | Key | Description |
| :--- | :--- | :--- |
| Training | `training.epochs` | Number of full epochs. |
| Training | `training.batch_size` | Batch size per step. |
| Training | `training.learning_rate` | AdamW learning rate. |
| Training | `training.loss_type` | `cross_entropy`, `weighted_cross_entropy`, or `focal`. |
| Model | `model.num_experts` | Number of MoE experts. |
| Model | `model.top_k` | Experts selected per token. |
| General | `seed` | Random seed. |
| Logging | `logging.use_wandb` | Enable or disable W&B. |

Use `--set section.key=value` to override values without editing `configs/vihsd.yaml`.